In [1]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import archnemesis as ans
from archnemesis.database.datatypes.wave_range import WaveRange


# Define files used later

test_data_dir = Path("./test_data")
ANS_LINE_DATABASE_FILE = test_data_dir / "test_ld.h5"
ANS_CONTINUUM_DATABASE_FILE = test_data_dir / "test_pc.h5"
ANS_PARTITION_FUNCTION_DATABASE_FILE = test_data_dir / "hitran24_pf.h5"

In [2]:
gasID, isoID, ambient_gas = (
    ans.enums.Gas.CH4, 
    0,
    ans.enums.AmbientGas.AIR,
)
vmin, vmax = WaveRange(1.3, 3.5, ans.enums.WaveUnit.Wavelength_um).as_unit(ans.enums.WaveUnit.Wavenumber_cm).values()

print(f'{gasID=} {isoID=}')
print(f'{vmin=} {vmax=}')

gasID=<Gas.CH4: 6> isoID=0
vmin=2857.1428571428573 vmax=7692.3076923076915


In [3]:
# Create a LineData_0 instance and load the data we need

TEMPERATURE = 300

line_data = ans.LineData_0(
	ID=gasID,
	ISO=isoID,
	ambient_gasses=ambient_gas,
	LINE_DATABASE = ANS_LINE_DATABASE_FILE,
	CONTINUUM_DATABASE = ANS_CONTINUUM_DATABASE_FILE,
	PARTITION_FUNCTION_DATABASE = ANS_PARTITION_FUNCTION_DATABASE_FILE,
)

print(f'{line_data._params_fetched_lines_last=}')
print(f'{line_data._params_fetched_partition_last=}')

# Loading the line data in our required spectral range
line_data.set_params(vmin=vmin,vmax=vmax, temperature=TEMPERATURE).fetch_linedata()

# Loading the partition function data
line_data.fetch_partition_fn()

print(f'{line_data._params_fetched_lines_last=}')
print(f'{line_data._params_fetched_partition_last=}')

INFO :: get_data :: ans_pseudo_continuum_file.py-493 :: Found compatible data for s_max=np.float64(1e-26) temperature=np.float64(300.0). Chosen group has leaf_grp_pc_parameters=(np.float64(1e-26), np.float64(300.0), np.int64(1))


line_data._params_fetched_lines_last=False
line_data._params_fetched_partition_last=False
self._params=LineDataParams(ambient_gasses=(<AmbientGas.AIR: 0>,), wn_min=2857.1428571428573, wn_max=7692.3076923076915, s_min=-1, temperature=300, pressure=0)
self.mol_ids=array([6])
self.iso_ids=array([[1, 2, 3, 4]])
TESTING: (s_max, temperature)=(np.float64(1e-26), np.float64(300.0)) leaf_grp_pc_parameters=(np.float64(1e-26), np.float64(200.0), np.int64(1))
TESTING: (s_max, temperature)=(np.float64(1e-26), np.float64(300.0)) leaf_grp_pc_parameters=(np.float64(1e-26), np.float64(250.0), np.int64(1))
TESTING: (s_max, temperature)=(np.float64(1e-26), np.float64(300.0)) leaf_grp_pc_parameters=(np.float64(1e-26), np.float64(300.0), np.int64(1))
type(id_line_cont_triplet)=<class 'list'> len(id_line_cont_triplet)=4
type(id_line_cont_triplet[0])=<class 'tuple'> len(id_line_cont_triplet[0])=3
type(id_line_cont_triplet[0][0])=<class 'tuple'> len(id_line_cont_triplet[0][0])=2
single_iso_line_set_data.nu=a

In [ ]:
# Calculate line strengths and plot them #

line_strengths = line_data.set_params(temperature=TEMPERATURE).calculate_line_strength()

for x in zip(line_data.line_data, line_strengths):
	plt.plot(x.NU, line_strengths)
plt.title(f'Line strength for ({line_data.ID}, {line_data.ISO})')
plt.xlabel('Wavenumber (cm^{-1})')
plt.ylabel('Line Strength (cm^{-1} / [molec cm^{-2}])')
plt.show()

AttributeError: 'LineData_0' object has no attribute 'calculate_line_strength'

In [ ]:
# Define a set of wavelengts to calculate absorption coefficient at and then plot them #

waves = np.linspace(2800, 3200, 1000)
k_abs = line_data.calculate_monochromatic_absorption(waves, 200, 1)

plt.plot(waves, k_abs)
plt.title(f'Absorption Coefficient for ({line_data.ID}, {line_data.ISO})')
plt.xlabel('Wavenumber (cm^{-1})')
plt.ylabel('Absorption Coefficient (1 / [molec cm^{-2}])')
plt.show()

In [ ]:
# Create example plot #

line_data.plot_linedata()